In [1]:
import os

In [2]:
%pwd

'd:\\projects\\capstone_project\\deep_learning_projects\\Kidney-Disease-Classification-Deep-Learning-Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\projects\\capstone_project\\deep_learning_projects\\Kidney-Disease-Classification-Deep-Learning-Project'

In [5]:
os.environ["MLFLOW_TRACKING_URI"]=os.getenv("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_USERNAME"]=os.getenv("MLFLOW_TRACKING_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"]=os.getenv("MLFLOW_TRACKING_PASSWORD")

In [6]:
import tensorflow as tf

In [7]:
model = tf.keras.models.load_model("artifacts/training/model.keras")

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [ ]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        mlflow_uri = os.getenv(
            "MLFLOW_TRACKING_URI",
            "https://dagshub.com/dnyaneshwarmagar/Chest-Cancer-Classification-Project.mlflow",
        )
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.keras",
            training_data="artifacts/data_ingestion/Chest-CT-Scan-data",
            mlflow_uri=mlflow_uri,
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config


In [11]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.tensorflow
from urllib.parse import urlparse


In [ ]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        image_size = tuple(self.config.params_image_size[:-1])

        self.valid_ds = tf.keras.utils.image_dataset_from_directory(
            directory=str(self.config.training_data),
            validation_split=0.30,
            subset="validation",
            seed=42,
            image_size=image_size,
            batch_size=self.config.params_batch_size,
            label_mode="categorical",
            interpolation="bilinear",
            shuffle=False,
        )
        self.valid_ds = self.valid_ds.map(
            lambda image, label: (tf.cast(image, tf.float32) / 255.0, label),
            num_parallel_calls=tf.data.AUTOTUNE,
        ).prefetch(tf.data.AUTOTUNE)

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_ds)
        self.save_score()

    def save_score(self):
        scores = {"loss": float(self.score[0]), "accuracy": float(self.score[1])}
        save_json(path=Path("scores.json"), data=scores)

    @staticmethod
    def _should_register_model(tracking_uri: str) -> bool:
        register_flag = os.getenv("MLFLOW_REGISTER_MODEL", "false").strip().lower() in {
            "1",
            "true",
            "yes",
        }
        is_dagshub = "dagshub.com" in tracking_uri.lower()
        return register_flag and not is_dagshub

    def log_into_mlflow(self):
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        tracking_uri = mlflow.get_tracking_uri()
        tracking_url_type_store = urlparse(tracking_uri).scheme
        print(f"MLflow tracking URI: {tracking_uri} (scheme={tracking_url_type_store})")

        register_model = self._should_register_model(tracking_uri)
        registered_model_name = os.getenv("MLFLOW_REGISTERED_MODEL_NAME", "VGG16Model")

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": float(self.score[0]), "accuracy": float(self.score[1])}
            )

            if register_model:
                mlflow.tensorflow.log_model(
                    self.model,
                    artifact_path="model",
                    registered_model_name=registered_model_name,
                )
                print(f"Model logged and registered as '{registered_model_name}'")
            else:
                mlflow.tensorflow.log_model(self.model, artifact_path="model")
                print("Model logged to MLflow as a run artifact (not registered)")


In [13]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

[2026-11-19 12:54:13,420: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-11-19 12:54:13,429: INFO: common: yaml file: params.yaml loaded successfully]
[2026-11-19 12:54:13,433: INFO: common: created directory at: artifacts]
Found 343 files belonging to 2 classes.
Using 102 files for validation.
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 1.0000 - loss: 0.0000e+00
[2026-11-19 12:54:22,997: INFO: common: json file saved at: scores.json]


2026/11/19 12:54:27 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/11/19 12:54:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run brawny-sloth-226 at: https://dagshub.com/dnyaneshwarmagar/Chest-Cancer-Classification-Project.mlflow/#/experiments/0/runs/c3ec597390f140ee80ac6d1ae83e0a37
🧪 View experiment at: https://dagshub.com/dnyaneshwarmagar/Chest-Cancer-Classification-Project.mlflow/#/experiments/0


MlflowException: API request to endpoint /api/2.0/mlflow/registered-models/create failed with error code 404 != 200. Response body: ''